## Modelo XGBoost

### Carga de librerías

In [1]:
import polars as pl
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import log_loss
import xgboost as xgb
from pathlib import Path

### Carga de datos

In [ ]:
ROOT = Path("..")
ruta_t1 = ROOT / "data" / "processed" / "temporada1_limpio.parquet"
temporada1 = pl.read_parquet(ruta_t1).to_pandas()

Vamos a chequear las columnas que tiene el dataset de la temporada 2. Las columnas a incluir en el modelo XGBoost van a ser todas las que contenga el dataset de la temporada 2 más las que se calculan a partir de ellas (*feature engineering*)

La única diferencia entre los archivos .parquet de las temporadas 1 y 2 son las variables *swing* y *description* (variable respuesta) y todas las que calculamos en el *feature engineering*.

### Definición de variables predictoras y respuesta

In [5]:
columnas_excluir = ['pitch_id', 'batter', 'pitcher', 'description', 'swing']
columnas_contaminadas = [
    'sz_top', 'sz_bot', 'sz_mid', 'strike_zone_size', # Los sensores directos
    'relative_height', 'is_strike_zone', 'is_shadow_zone', # Calculadas en el featuring engineering
    'pitch_location', 'distance_to_corner'
]

todas_excluidas = columnas_excluir + columnas_contaminadas

explicativas = temporada1.drop(columns=[col for col in todas_excluidas if col in temporada1.columns])
respuesta = temporada1['swing']

# A XGBoost hay que definirle cuáles son las variables categóricas
columnas_texto = explicativas.select_dtypes(include=['object', 'string']).columns
for col in columnas_texto:
    explicativas[col] = explicativas[col].astype('category')

print(f"El dataset tiene {explicativas.shape[0]} filas y {explicativas.shape[1]} variables predictoras.")

El dataset tiene 704721 filas y 15 variables predictoras.


### Configuración del modelo XGBoost

In [6]:
xgb_base = xgb.XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    enable_categorical=True, # Avisa que hay columnas categóricas
    tree_method='hist',      # Algoritmo de histogramas, rápido para datasets grandes
    random_state=714
)

Vamos a definir una serie de hiperparámetros distintos y vamos a probar solo 10 al azar. Queda para otro momento el analizar otras técnicas de selección de hiperparámetros.

In [7]:
param_dist = {
    'max_depth': [3, 5, 7],               # Profundidad del árbol
    'learning_rate': [0.01, 0.05, 0.1],   # Qué tan rápido aprende
    'n_estimators': [100, 300, 500],      # Cantidad de árboles
    'subsample': [0.8, 1.0],              # Porcentaje de filas a usar por árbol
    'colsample_bytree': [0.8, 1.0]        # Porcentaje de columnas a usar por árbol
}

skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42) # Usamos 3 folds para que sea más rápido
random_search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_dist,
    n_iter=10, # límite de combinaciones de hiperparámetros a probar
    scoring='neg_log_loss', # Scikit-learn busca maximizar, por eso usa LogLoss negativo
    cv=skf,
    verbose=1, # Te va mostrando el progreso en la consola
    random_state=569
)

random_search.fit(explicativas, respuesta)
print(f"Mejores hiperparámetros: {random_search.best_params_}")
print(f"Mejor LogLoss estimado: {-random_search.best_score_: .4f}")

mejor_modelo = random_search.best_estimator_

Fitting 3 folds for each of 10 candidates, totalling 30 fits
Mejores hiperparámetros: {'subsample': 1.0, 'n_estimators': 500, 'max_depth': 5, 'learning_rate': 0.1, 'colsample_bytree': 0.8}
Mejor LogLoss estimado:  0.4472


### Predicción para Kaggle

Primero, leemos los datos limpios de la temporada 2.

In [8]:
ruta_t2 = ROOT / "data" / "processed" / "temporada2_limpio.parquet"

temporada2= pl.read_parquet(ruta_t2)

In [11]:
temporada2 = temporada2.to_pandas()

# Guardamos los IDs para el archivo final (Kaggle siempre pide el ID)
pitch_ids_t2 = temporada2['pitch_id']

# Filtramos X_test para que tenga EXACTAMENTE las mismas columnas que usamos para entrenar
X_test = temporada2.drop(columns=[col for col in todas_excluidas if col in temporada2.columns])

# Hay un error de categoría 1-3. Lo convertimos en nulo.
X_test['count'] = X_test['count'].replace('1-3', np.nan)

# Pasamos las variables necesarias a categóricas, con las mismas categorías que el training set
columnas_texto = explicativas.select_dtypes(include=['category']).columns
for col in columnas_texto:
    X_test[col] = pd.Categorical(X_test[col], categories=explicativas[col].cat.categories)


# Predecimos las probabilidades
probabilidades_swing = mejor_modelo.predict_proba(X_test)[:, 1]

# Armamos el DataFrame final
predicciones = pl.DataFrame({
    'pitch_id': pitch_ids_t2,
    'swing_probability': probabilidades_swing
})

In [12]:
# Guardamos el archivo
ruta_prediccion = ROOT / "reports" / "validacion.parquet"
predicciones.write_parquet(ruta_prediccion)